In [1]:
import duckdb, time, hashlib
con = duckdb.connect()
full_path = 'C:/Users/alber/Downloads/archive (1)/itineraries.csv'
csv_path = 'C:/Users/alber/Downloads/archive (1)/itineraries_sample.csv'

In [2]:
t0 = time.time()
with open(full_path, 'r', encoding='utf-8') as infile, open(csv_path, 'w', encoding='utf-8') as outfile:
    for i, line in enumerate(infile, start=1):
        if i == 1 or i % 2 == 0:
            outfile.write(line)
elapsed = time.time() - t0
print(f"Sampling: {elapsed:.2f}s")

Sampling: 169.05s


In [3]:
t0 = time.time()
h = hashlib.md5()
with open(csv_path, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''):
        h.update(chunk)
print("MD5:", h.hexdigest())
print(f"Checksum time: {time.time()-t0:.2f}s")

MD5: 77038bbfcda7a7b5f53c8b760f225a0b
Checksum time: 39.98s


In [4]:
with open(csv_path, 'r', encoding='utf-8') as f:
    n = sum(1 for _ in f)
print(f"Rows (incl. header): {n}")

Rows (incl. header): 41069378


In [5]:
t0 = time.time()
con.execute("DROP TABLE IF EXISTS flights")
con.execute(f"CREATE TABLE flights AS SELECT * FROM '{csv_path}'")
elapsed = time.time() - t0
print(f"Table build: {elapsed:.4f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Table build: 182.9338s


In [6]:
t0 = time.time()
result = con.execute("""
    SELECT * FROM flights WHERE legId = '9335fae376c38bb61263281779f469ec'
""").fetchall()
elapsed = time.time() - t0
print(len(result), "rows", f"{elapsed:.4f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows 8.6369s


In [7]:
t0 = time.time()
result = con.execute("""
    SELECT * FROM flights
    WHERE startingAirport='JFK' AND destinationAirport='LAX'
      AND flightDate BETWEEN '2022-06-01' AND '2022-06-30'
      AND totalFare < 300
""").fetchall()
elapsed = time.time() - t0
print(len(result), "rows", f"{elapsed:.4f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3421 rows 29.7747s


In [8]:
t0 = time.time()
result = con.execute("""
    SELECT startingAirport, destinationAirport,
           STRFTIME(flightDate, '%Y-%m') AS month,
           AVG(totalFare) AS avg_fare, COUNT(*) AS n
    FROM flights
    GROUP BY startingAirport, destinationAirport, month
    ORDER BY month
""").fetchall()
elapsed = time.time() - t0
print(len(result), "rows", f"{elapsed:.4f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1873 rows 10.0253s


In [9]:
t0 = time.time()
result = con.execute("""
    SELECT startingAirport, destinationAirport, isNonStop,
           AVG(totalFare) AS avg_fare
    FROM flights
    GROUP BY startingAirport, destinationAirport, isNonStop
""").fetchall()
elapsed = time.time() - t0
print(len(result), "rows", f"{elapsed:.4f}s")

450 rows 1.6579s


In [10]:
t0 = time.time()
con.execute("""
    INSERT INTO flights (legId, searchDate, flightDate, startingAirport, destinationAirport,
      fareBasisCode, travelDuration, elapsedDays, isBasicEconomy, isRefundable, isNonStop,
      baseFare, totalFare, seatsRemaining, totalTravelDistance,
      segmentsDepartureTimeEpochSeconds, segmentsDepartureTimeRaw,
      segmentsArrivalTimeEpochSeconds, segmentsArrivalTimeRaw,
      segmentsArrivalAirportCode, segmentsDepartureAirportCode,
      segmentsAirlineName, segmentsAirlineCode, segmentsEquipmentDescription,
      segmentsDurationInSeconds, segmentsDistance, segmentsCabinCode)
    VALUES ('synthetic-tier3-test', '2022-08-01', '2022-08-02', 'JFK', 'LAX',
      'TESTCODE', 'PT6H', 0, 0, 0, 1, 300.00, 350.00, 10, 2475,
      '1658812800', '2022-08-02T08:00:00.000-04:00',
      '1658827200', '2022-08-02T12:00:00.000-04:00',
      'LAX', 'JFK', 'Delta', 'DL',
      'Boeing 737-800', '14400', '2475', 'coach')
""")
elapsed = time.time() - t0
print(f"{elapsed:.4f}s")

0.1141s


In [11]:
t0 = time.time()
con.execute("""
    UPDATE flights SET seatsRemaining = seatsRemaining - 1
    WHERE flightDate = '2022-07-04' AND startingAirport = 'ATL'
""")
elapsed = time.time() - t0
print(f"{elapsed:.4f}s")

0.3028s
